# 天气图片分类 - 智海算法调优
## 四类天气识别: cloudy(阴天) / rain(雨天) / shine(晴天) / sunrise(日出)

**数据集**: Kaggle Multi-class Weather Dataset (MWD)
**评分规则**: 最终得分 = F1分数 × 100 | **同分比较**: 模型推理时间

### Kaggle MWD 部署步骤:
1. 从 Kaggle 下载数据集 → 上传到Mo平台 → 解压
2. 从 Step 0 开始依次运行每个 Cell
3. 训练完成后部署应用

---
## 准备数据 — 从 Kaggle 下载 MWD 数据集

### Step A: 在Kaggle下载
访问 https://www.kaggle.com/datasets/saurabhshahane/multi-class-weather-dataset
点击 **Download** 下载 `archive.zip`

### Step B: 上传到Mo平台
在Mo平台左侧文件栏 → 上传按钮 → 选择 `archive.zip`

### Step C: 解压并整理
运行下方 Cell 解压数据。MWD数据集解压后结构：
```
archive/
  cloudy/     ← 300张阴天图片
  rain/       ← 215张雨天图片
  shine/      ← 253张晴天图片
  sunrise/    ← 357张日出图片
```
代码会自动按此结构读取。

## Step 0: 安装依赖

In [ ]:
!pip install torch torchvision scikit-learn Pillow -q

In [ ]:
# ★ 解压 Kaggle MWD 数据集
# 先确认上传的压缩包文件名，修改下面这行
import zipfile, os

ZIP_FILE = "archive.zip"   # ← 改成你实际上传的文件名

if os.path.exists(ZIP_FILE):
    print(f"正在解压 {ZIP_FILE} ...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zf:
        zf.extractall("./")
    print("解压完成!")
    # 列出解压后的目录结构
    for item in os.listdir("."):
        if os.path.isdir(item):
            cnt = len([f for f in os.listdir(item) if f.endswith(('.jpg','.jpeg','.png'))])
            if cnt > 0:
                print(f"  {item}/ ({cnt} images)")
else:
    print(f"未找到 {ZIP_FILE}")
    print("请先在左侧文件栏上传Kaggle下载的 archive.zip")
    print("或修改 ZIP_FILE 变量为你的文件名")
    # 列出当前目录所有文件和文件夹帮助排查
    print("\n当前目录内容:")
    for item in sorted(os.listdir(".")):
        print(f"  {item}")

## Step 1: 配置参数
> GPU环境建议: `NUM_EPOCHS=60, BATCH_SIZE=64`

In [ ]:
import os, sys, time, datetime
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms, models
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

# ======== 超参数 ========
IMAGE_SIZE = 224
BATCH_SIZE = 64          # GPU用64
NUM_EPOCHS = 60          # GPU训练60轮
WARMUP_EPOCHS = 3
LEARNING_RATE = 1e-3
FINE_TUNE_LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.3
CUTMIX_ALPHA = 0.2
MIXUP_PROB = 0.5
GRAD_CLIP = 1.0
# ★ Kaggle MWD 数据集四分类
CLASS_NAMES = ["cloudy", "rain", "shine", "sunrise"]
NUM_CLASSES = len(CLASS_NAMES)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Classes: {CLASS_NAMES}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")

## Step 2: 数据增强

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25),
])

val_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("数据增强配置完成")

class WeatherDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
        if os.path.isdir(root_dir):
            for class_name in CLASS_NAMES:
                class_dir = os.path.join(root_dir, class_name)
                if os.path.isdir(class_dir):
                    for fname in os.listdir(class_dir):
                        if fname.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp')):
                            self.samples.append((os.path.join(class_dir, fname),
                                                 self.class_to_idx[class_name]))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0,0,0))
        if self.transform:
            image = self.transform(image)
        return image, label


# ★ 自动查找数据目录（适配Kaggle MWD解压后的各种结构）
def find_data_dir():
    """自动查找包含 cloudy/rain/shine/sunrise 子目录的路径"""
    candidates = ["./data/train", "./data", "./archive", "./dataset", "."]
    for path in candidates:
        if os.path.isdir(path):
            found = sum(1 for cls in CLASS_NAMES if os.path.isdir(os.path.join(path, cls)))
            if found >= 4:
                return path
    # 没找到, 扫描子目录
    for root, dirs, files in os.walk("."):
        found = sum(1 for cls in CLASS_NAMES if cls in dirs)
        if found >= 4:
            return root
    return "./data/train"  # 默认

DATA_PATH = find_data_dir()
print(f"数据目录: {DATA_PATH}")

train_dataset = WeatherDataset(DATA_PATH, transform=train_transform)
print(f"训练集: {len(train_dataset)} 张")
for cls in CLASS_NAMES:
    cnt = sum(1 for _, l in train_dataset.samples if CLASS_NAMES[l] == cls)
    print(f"  {cls}: {cnt} 张")

if len(train_dataset) == 0:
    print("\n⚠️ 未找到数据！请检查:")
    print("1. 是否已运行解压Cell?")
    print("2. 解压后目录是否包含 cloudy/ rain/ shine/ sunrise/ 四个文件夹?")
    print("3. 当前目录内容:")
    for item in sorted(os.listdir(".")):
        print(f"   {item}")

# 从训练集切分15%验证集
from torch.utils.data import random_split
val_size = max(1, int(0.15 * len(train_dataset)))
train_size = len(train_dataset) - val_size
train_dataset, val_dataset = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)
val_dataset.dataset.transform = val_transform
print(f"切分: train={train_size}, val={val_size}")

In [ ]:
class WeatherDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
        if os.path.isdir(root_dir):
            for class_name in CLASS_NAMES:
                class_dir = os.path.join(root_dir, class_name)
                if os.path.isdir(class_dir):
                    for fname in os.listdir(class_dir):
                        if fname.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp')):
                            self.samples.append((os.path.join(class_dir, fname),
                                                 self.class_to_idx[class_name]))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0,0,0))
        if self.transform:
            image = self.transform(image)
        return image, label


DATA_PATH = "./data/"
train_path = os.path.join(DATA_PATH, "train")
val_path = os.path.join(DATA_PATH, "val")

train_dataset = None
val_dataset = None

if os.path.isdir(train_path):
    train_dataset = WeatherDataset(train_path, transform=train_transform)
    print(f"训练集: {len(train_dataset)} 张")
    for cls in CLASS_NAMES:
        cnt = sum(1 for _, l in train_dataset.samples if CLASS_NAMES[l] == cls)
        print(f"  {cls}: {cnt} 张")
else:
    print(f"未找到训练集: {train_path}")
    print("请在左侧文件栏创建 data/train/ 文件夹并上传图片!")

if os.path.isdir(val_path):
    val_dataset = WeatherDataset(val_path, transform=val_transform)
    print(f"验证集: {len(val_dataset)} 张")
elif train_dataset is not None:
    from torch.utils.data import random_split
    val_size = max(1, int(0.15 * len(train_dataset)))
    train_size = len(train_dataset) - val_size
    train_dataset, val_dataset = random_split(
        train_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42),
    )
    print(f"从训练集切分: train={train_size}, val={val_size}")

## Step 4: 数据加载器

In [ ]:
if train_dataset is not None:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True) if val_dataset else None
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader) if val_loader else 0}")

## Step 5: 构建模型 (EfficientNet-B0 + 增强分类头)
> 5.3M参数，ImageNet预训练，分类头 1280→256→4

In [ ]:
def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.35),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, NUM_CLASSES),
    )
    return model.to(DEVICE)


model = build_model()
total_p = sum(p.numel() for p in model.parameters()) / 1e6
print(f"EfficientNet-B0: {total_p:.2f}M params | 分类头: 1280→256→4")
print(f"类别: {CLASS_NAMES}")

## Step 6: MixUp / CutMix

In [ ]:
def mixup_data(x, y, alpha=0.3):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    index = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[index], y, y[index], lam


def cutmix_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    B, _, H, W = x.size()
    index = torch.randperm(B, device=x.device)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bw, bh = int(W * np.sqrt(1 - lam)), int(H * np.sqrt(1 - lam))
    x0, y0 = np.clip(cx - bw // 2, 0, W), np.clip(cy - bh // 2, 0, H)
    x1, y1 = np.clip(cx + bw // 2, 0, W), np.clip(cy + bh // 2, 0, H)
    mixed_x = x.clone()
    mixed_x[:, :, y0:y1, x0:x1] = x[index, :, y0:y1, x0:x1]
    lam = 1 - ((x1 - x0) * (y1 - y0) / (H * W))
    return mixed_x, y, y[index], lam

print("MixUp + CutMix 就绪")

## Step 7: 训练函数

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0
    amp_enabled = torch.cuda.is_available()
    
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        
        # MixUp / CutMix
        if epoch >= WARMUP_EPOCHS and torch.rand(1).item() < MIXUP_PROB:
            if torch.rand(1).item() < 0.6:
                images, targets_a, targets_b, lam = mixup_data(images, targets, MIXUP_ALPHA)
            else:
                images, targets_a, targets_b, lam = cutmix_data(images, targets, CUTMIX_ALPHA)
            mixup_active = True
        else:
            mixup_active = False
        
        optimizer.zero_grad(set_to_none=True)
        
        if amp_enabled:
            with autocast():
                outputs = model(images)
                if mixup_active:
                    loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
                else:
                    loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            if mixup_active:
                loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
            else:
                loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, targets)
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
    
    return {
        "loss": total_loss / len(loader),
        "accuracy": accuracy_score(all_targets, all_preds),
        "f1_macro": f1_score(all_targets, all_preds, average="macro"),
        "f1_weighted": f1_score(all_targets, all_preds, average="weighted"),
    }

print("训练函数就绪")

## Step 8: 开始训练
> 两阶段训练: WarmUp(仅分类头) → FineTune(全模型+CosWarmRestarts)

In [ ]:
if train_dataset is None:
    print("请先运行 Step 3 加载数据集!")
else:
    print("=" * 60)
    print(f"训练 weather classifier | GPU: {torch.cuda.is_available()}")
    print(f"Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | 训练集: {len(train_dataset)}")
    print("=" * 60)
    
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    
    # Phase 1: 冻结 backbone
    for name, param in model.named_parameters():
        param.requires_grad = "classifier" in name
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    best_f1 = 0.0
    start_time = time.time()
    
    for epoch in range(NUM_EPOCHS):
        # Phase 2: 解冻 backbone
        if epoch == WARMUP_EPOCHS:
            print(f"\n>>> Epoch {epoch+1}: 解冻backbone，全模型微调")
            for param in model.parameters():
                param.requires_grad = True
            head_p = [p for n, p in model.named_parameters() if "classifier" in n and p.requires_grad]
            back_p = [p for n, p in model.named_parameters() if "classifier" not in n and p.requires_grad]
            optimizer = optim.AdamW([
                {"params": head_p, "lr": LEARNING_RATE},
                {"params": back_p, "lr": FINE_TUNE_LR},
            ], weight_decay=WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=5, T_mult=2, eta_min=1e-6)
        
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        
        if val_loader:
            val_m = validate(model, val_loader, criterion)
            print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | loss={train_loss:.4f} | "
                  f"v_acc={val_m['accuracy']:.4f} | v_f1={val_m['f1_macro']:.4f} | lr={lr:.2e}")
            
            if val_m['f1_macro'] > best_f1:
                best_f1 = val_m['f1_macro']
                torch.save({
                    "epoch": epoch, "model_state_dict": model.state_dict(),
                    "class_names": CLASS_NAMES, "image_size": IMAGE_SIZE,
                    "val_f1": best_f1, "val_acc": val_m['accuracy'],
                }, "best_model.pth")
                print(f"  >>> Best model (F1={best_f1:.4f}, score={best_f1*100:.1f})")
        else:
            print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | loss={train_loss:.4f} | lr={lr:.2e}")
    
    elapsed = time.time() - start_time
    print(f"\n训练完成! 耗时: {elapsed/60:.1f}min | 最佳F1: {best_f1:.4f} | 得分: {best_f1*100:.1f}")

## Step 9: 推理测试

In [ ]:
@torch.no_grad()
def predict_image(image_path, model_path="best_model.pth"):
    model = build_model()
    if os.path.exists(model_path):
        ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    model.eval()
    
    image = Image.open(image_path).convert("RGB")
    tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    
    t0 = time.time()
    output = model(tensor)
    probs = torch.softmax(output, dim=1).cpu().numpy()[0]
    t = (time.time() - t0) * 1000
    
    pred_idx = int(np.argmax(probs))
    print(f"预测: {CLASS_NAMES[pred_idx]} | 推理: {t:.1f}ms")
    for i, (n, p) in enumerate(zip(CLASS_NAMES, probs)):
        print(f"  {n}: {p*100:5.1f}%{' <--' if i == pred_idx else ''}")
    return CLASS_NAMES[pred_idx], probs


print("推理函数就绪")
# 示例: predict_image("./test.jpg")

## Step 10: Handle 函数（应用部署接口）
> **这是部署的核心函数**，Mo平台通过它识别输入输出参数。

In [ ]:
def handle(image_path, model_path="best_model.pth"):
    """
    天气分类应用入口
    输入:
        image_path: str  - 图片文件路径
        model_path: str  - 模型权重路径 (默认 best_model.pth)
    输出:
        {
            "prediction": str,          - 天气类别
            "confidence": float,         - 置信度
            "probabilities": dict,       - 各类别概率
            "inference_time_ms": float   - 推理耗时
        }
    """
    import time as _t
    
    model = build_model()
    if os.path.exists(model_path):
        ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    model.eval()
    
    image = Image.open(image_path).convert("RGB")
    tensor = val_transform(image).unsqueeze(0)
    
    start = _t.time()
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]
    inference_time = (_t.time() - start) * 1000
    
    pred_idx = int(np.argmax(probs))
    return {
        "prediction": CLASS_NAMES[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
        "probabilities": {n: round(float(p), 4) for n, p in zip(CLASS_NAMES, probs)},
        "inference_time_ms": round(inference_time, 2),
    }


print("Handle函数就绪，可用于部署")
print("输入: image_path | 输出: prediction, confidence, probabilities, inference_time_ms")